# Notebook 11 — Objectif 2 : Environnement × comportement (version finale)

**Conforme au SOW, Objectif 2.** Remplace les notebooks 08/09 en utilisant le fichier officiel
daté `Scan_Tot_newVersion_SMN.xlsx`.

**Tâche 2.1 — Synchronisation :** fusionner trois sources sur une base temporelle commune :
- IceTag (activité, bins de 15 min) ;
- HOBO (température, humidité, THI calculé) ;
- scans comportementaux (composition, désormais datés par vache).

**Tâche 2.2 — Analyse exploratoire :** relier (a) l'activité IceTag et (b) la composition
comportementale aux conditions environnementales (THI), sur le corpus Summer 2019 (le plus riche
en HOBO et en variation thermique).

**Livrables SOW :** dataset intégré + documentation de synchronisation (2.1) ; visualisations +
rapport exploratoire (2.2).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import glob
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = PROJECT / 'Données completes' / 'Données accelerometres'
REPORTS = PROJECT / 'reports' / 'objective1_pipeline_icetag'
OUT = PROJECT / 'reports' / 'objective2_environnement'
OUT.mkdir(parents=True, exist_ok=True)

ICETAG_INPUT = REPORTS / 'summer_2019_pipeline_input_15min.csv'
HOBO_DIR = DATA_ROOT / 'Summer 2019' / 'Hobo'
SCANS = PROJECT / 'Données completes' / 'Scan_Tot_newVersion_SMN.xlsx'
BEHAV = ['Pct_locomotion', 'Pct_lying', 'Pct_Idle', 'Pct_Vigilance',
         'Pct_Explo', 'Pct_eating', 'Pct_Social', 'Pct_Maintenance', 'Pct_Other']
print('OK' if all(p.exists() for p in [ICETAG_INPUT, HOBO_DIR, SCANS]) else 'MANQUANT')

OK


## 1. Environnement (HOBO) → THI à 15 min

In [2]:
hobo_files = [h for h in glob.glob(str(HOBO_DIR / '**' / '*.xlsx'), recursive=True)
              if 'THI' not in h and ('Outside' in h or 'outside' in h)]

def parse_hobo(f):
    df = pd.read_excel(f, header=1)
    dcol = [c for c in df.columns if 'Date' in str(c)][0]
    tcol = [c for c in df.columns if str(c).startswith('Temp.,')][0]
    hcol = [c for c in df.columns if str(c).startswith('HR,')][:1]
    df = df[[dcol, tcol] + hcol].copy()
    df.columns = ['dt', 'temp', 'rh'][:1 + 1 + len(hcol)]
    df['dt'] = pd.to_datetime(df['dt'], errors='coerce')
    return df.dropna(subset=['dt'])

env = pd.concat([parse_hobo(f) for f in hobo_files], ignore_index=True)
env = env.drop_duplicates('dt').sort_values('dt')
env['THI'] = (1.8 * env['temp'] + 32) - (0.55 - 0.0055 * env['rh']) * (1.8 * env['temp'] - 26)

def thi_cat(t):
    return '1_aucun' if t < 68 else '2_leger' if t < 72 else '3_modere' if t < 80 else '4_severe'

env15 = env.set_index('dt')[['temp', 'rh', 'THI']].resample('15min').mean().dropna().reset_index()
env_daily = env.set_index('dt')[['temp', 'rh', 'THI']].resample('1D').mean().dropna()
env_daily.columns = ['temp_jour', 'rh_jour', 'THI_jour']
print(f"HOBO : {len(env15)} bins 15 min | {len(env_daily)} jours")
print(f"Température {env['temp'].min():.0f}–{env['temp'].max():.0f} °C, THI {env['THI'].min():.0f}–{env['THI'].max():.0f}")

HOBO : 6066 bins 15 min | 65 jours
Température 8–33 °C, THI 47–84


## 2. Tâche 2.1 — Dataset intégré (IceTag + environnement)

In [3]:
act = pd.read_csv(ICETAG_INPUT)
act['Cow'] = act['Cow'].astype(str)
act['Start'] = pd.to_datetime(act['Start'])
icetag_env = act.merge(env15, left_on='Start', right_on='dt', how='inner')
icetag_env.to_csv(OUT / 'summer2019_icetag_environnement_15min.csv', index=False)
print(f"IceTag × environnement : {len(icetag_env)} bins, {icetag_env['Cow'].nunique()} vaches")

IceTag × environnement : 87501 bins, 17 vaches


## 3. Tâche 2.1 — Dataset intégré (comportement + environnement)

Les scans étant datés, on attache à chaque scan les conditions environnementales **du jour**.

In [4]:
scans = pd.read_excel(SCANS, sheet_name='Feuil1')
scans['Cow'] = scans['Cow'].astype(str).str.replace('.0', '', regex=False)
scans['Date'] = pd.to_datetime(scans['Date'], errors='coerce')
summer_sc = scans[(scans['Experiment'] == 'Summer2019') & scans['Date'].notna()].copy()
summer_sc['jour'] = summer_sc['Date'].dt.normalize()

behav_env = summer_sc.merge(env_daily, left_on='jour', right_index=True, how='inner')
behav_env.to_csv(OUT / 'summer2019_comportement_environnement.csv', index=False)
print(f"Scans comportementaux reliés à l'environnement du jour : {len(behav_env)} / {len(summer_sc)}")
print(f"Période : {behav_env['jour'].min().date()} -> {behav_env['jour'].max().date()}")
print(f"THI journalier des jours de scan : {behav_env['THI_jour'].min():.0f}–{behav_env['THI_jour'].max():.0f}")

Scans comportementaux reliés à l'environnement du jour : 51 / 58
Période : 2019-07-04 -> 2019-08-23
THI journalier des jours de scan : 63–73


## 4. Tâche 2.2 — Activité IceTag vs THI (rappel synthétique)

In [5]:
d = icetag_env[(icetag_env['Start'].dt.hour >= 6) & (icetag_env['Start'].dt.hour < 20)]
rho, p = spearmanr(d['THI'], d['Steps'])
print(f"Activité (pas) vs THI, bins de jour : rho = {rho:+.3f} (p = {p:.1e}, n = {len(d)})")

Activité (pas) vs THI, bins de jour : rho = +0.097 (p = 1.5e-106, n = 51258)


## 5. Tâche 2.2 — Composition comportementale vs THI (apport du fichier daté)

In [6]:
# L'environnement varie au niveau du JOUR : on agrège les comportements par jour
# (sinon pseudo-réplication : plusieurs vaches le même jour partagent le même THI).
daily_behav = behav_env.groupby('jour').agg(
    THI_jour=('THI_jour', 'first'),
    **{b: (b, 'mean') for b in BEHAV}
).dropna(subset=['THI_jour'])
print(f"Jours de scan distincts (unité d'analyse correcte) : {len(daily_behav)}")
print()
print('=== Corrélation (Spearman) THI du jour vs comportement, au niveau JOUR ===')
rows = []
for b in BEHAV:
    sub = daily_behav[['THI_jour', b]].dropna()
    if len(sub) < 5 or sub[b].nunique() < 3:
        continue
    rho, pval = spearmanr(sub['THI_jour'], sub[b])
    rows.append({'comportement': b.replace('Pct_', ''), 'rho_vs_THI': round(rho, 3),
                 'p': round(pval, 4), 'n_jours': len(sub),
                 'signif': 'OUI' if pval < 0.05 else 'non'})
res = pd.DataFrame(rows).sort_values('rho_vs_THI', ascending=False)
res.to_csv(OUT / 'summer2019_comportement_vs_THI.csv', index=False)
print(res.to_string(index=False))
print(f"\nNB : analyse sur {len(daily_behav)} jours seulement -> resultats suggestifs, faible puissance.")

Jours de scan distincts (unité d'analyse correcte) : 8

=== Corrélation (Spearman) THI du jour vs comportement, au niveau JOUR ===
comportement  rho_vs_THI      p  n_jours signif
      eating       0.738 0.0366        8    OUI
  locomotion      -0.098 0.8182        8    non
        Idle      -0.286 0.4927        8    non
      Social      -0.357 0.3851        8    non
 Maintenance      -0.359 0.3821        8    non
       Explo      -0.595 0.1195        8    non

NB : analyse sur 8 jours seulement -> resultats suggestifs, faible puissance.


## 6. Visualisation

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# (a) activité par tranche de THI
d2 = d.copy()
d2['THI_bin'] = pd.cut(d2['THI'], bins=[55, 65, 70, 75, 85])
tb = d2.groupby('THI_bin')['Steps'].mean()
axes[0].bar([str(i) for i in tb.index], tb.values, color='tomato')
axes[0].set_title('Activité (pas/15 min) vs THI'); axes[0].set_xlabel('THI'); axes[0].set_ylabel('Pas moyens')
axes[0].tick_params(axis='x', rotation=30)
# (b) comportements vs THI (locomotion, lying, idle, eating)
for b in ['Pct_locomotion', 'Pct_lying', 'Pct_Idle', 'Pct_eating']:
    sub = behav_env[['THI_jour', b]].dropna().sort_values('THI_jour')
    if len(sub) > 2:
        axes[1].scatter(sub['THI_jour'], sub[b], label=b.replace('Pct_', ''), alpha=0.6)
axes[1].set_title('Composition comportementale vs THI du jour')
axes[1].set_xlabel('THI journalier'); axes[1].set_ylabel('% du temps'); axes[1].legend(fontsize=8)
fig.suptitle('Summer 2019 — Environnement × comportement (fichier daté)', fontsize=13)
fig.tight_layout()
fig.savefig(OUT / 'summer2019_env_comportement_v2.png', dpi=120, bbox_inches='tight')
print('Figure sauvegardée.')
plt.show()

Figure sauvegardée.


## 7. Synthèse Objectif 2

In [8]:
rho_act, p_act = spearmanr(d['THI'], d['Steps'])
lines = []
lines.append('# Objectif 2 - Environnement x comportement (Summer 2019, fichier date)\n')
lines.append('## Tache 2.1 - Synchronisation')
lines.append(f'- IceTag x environnement : {len(icetag_env)} bins de 15 min, {icetag_env["Cow"].nunique()} vaches.')
lines.append(f'- Comportement x environnement : {len(behav_env)} scans datés ({len(daily_behav)} jours distincts) relies au THI du jour.')
lines.append('- Trois modalites synchronisees (activite, environnement, comportement).\n')
lines.append('## Tache 2.2 - Analyse')
lines.append(f'- Activite (pas) vs THI, niveau bin 15 min : rho = {rho_act:+.3f} (n={len(d)}). Association descriptive positive avant controle du jour; l effet intra-jour est non concluant dans l analyse de sensibilite finale.')
lines.append(f'- Composition comportementale vs THI, au niveau JOUR ({len(daily_behav)} jours) :')
lines.append(res.to_string(index=False))
lines.append('')
sig = res[res['signif'] == 'OUI']
if len(sig):
    for _, r in sig.iterrows():
        sens = 'augmente' if r['rho_vs_THI'] > 0 else 'diminue'
        lines.append(f"- {r['comportement']} {sens} avec le THI (rho = {r['rho_vs_THI']}, p = {r['p']}, sur {int(r['n_jours'])} jours).")
else:
    lines.append('- Aucun comportement ne varie significativement avec le THI au niveau jour.')
lines.append('\n## Limites (importantes)')
lines.append(f'- L\'analyse comportement-THI ne repose que sur {len(daily_behav)} jours de scan : '
             'puissance tres faible, resultats a considerer comme SUGGESTIFS et non concluants.')
lines.append('- Une analyse groupant les scans individuels (51) surestimerait la significativite '
             '(pseudo-replication : plusieurs vaches partagent le meme THI un jour donne). Le niveau '
             'jour est l\'unite correcte.')
lines.append('- Le stress thermique severe (THI >= 80) est rare dans ce corpus (climat quebecois).')
note = '\n'.join(lines)
(OUT / 'objectif2_synthese_finale.md').write_text(note, encoding='utf-8')
print(note)

# Objectif 2 - Environnement x comportement (Summer 2019, fichier date)

## Tache 2.1 - Synchronisation
- IceTag x environnement : 87501 bins de 15 min, 17 vaches.
- Comportement x environnement : 51 scans datés (8 jours distincts) relies au THI du jour.
- Trois modalites synchronisees (activite, environnement, comportement).

## Tache 2.2 - Analyse
- Activite (pas) vs THI, niveau bin 15 min : rho = +0.097 (n=51258). Association descriptive positive avant controle du jour; l'effet intra-jour est non concluant dans l'analyse de sensibilite finale.
- Composition comportementale vs THI, au niveau JOUR (8 jours) :
comportement  rho_vs_THI      p  n_jours signif
      eating       0.738 0.0366        8    OUI
  locomotion      -0.098 0.8182        8    non
        Idle      -0.286 0.4927        8    non
      Social      -0.357 0.3851        8    non
 Maintenance      -0.359 0.3821        8    non
       Explo      -0.595 0.1195        8    non

- eating augmente avec le THI (rho = 0.738, p